In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor

### Dagshub/Mlflow initialization ###

In [2]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='sbolk23', repo_name='House-Prices-Kaggle-Competition', mlflow=True)


Accessing as sbolk23

Initialized MLflow to track repo "sbolk23/House-Prices-Kaggle-Competition"

Repository sbolk23/House-Prices-Kaggle-Competition initialized!

In [3]:
PATH = '../data'
TARGET = 'SalePrice'

In [4]:
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

### Split Train/Test Data ###

In [5]:
df = pd.read_csv(PATH + '/train.csv')

In [6]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, shuffle=True, random_state=1337
)

train_ids = X_train['Id']
test_ids = X_test['Id']

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')

X_train shape (1168, 80)
X_test shape (292, 80)


### Data Cleaning ###

### Categorical Imputing ###

In [7]:
# we first impute all missing values in categorical and numeric features.

# These categorical columns may have 'NaN' values in them which means missing.
# data_description.txt did not mention 'Nan' was intended.

cat_impute_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'ExterQual',
    'ExterCond',
    'Heating',
    'HeatingQC',
    'CentralAir',
    'Electrical',
    'KitchenQual',
    'Functional',
    'PavedDrive',
    'SaleType',
    'SaleCondition',
]

### Numeric Imputing ###

In [8]:
# We impute all numeric columns because data_description.txt did not mention any numeric feature with intended 'NaN' value.

# num_impute_cols = [
#     'LotFrontage',
#     'GarageYrBlt',
#     'MasVnrArea',
# ]

num_impute_cols = list(X_train.select_dtypes(exclude='object').columns)
num_impute_cols.remove('Id')
num_impute_cols.remove('MSSubClass')

### Categorical One-Hot-Encoded Columns ###

In [9]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

cat_ohe_missing_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'PavedDrive',
    'SaleType',
    'SaleCondition'
]

cat_ohe_absent_cols = [
    'Alley',
    'MasVnrType',
    'GarageType',
    'MiscFeature'
]

### Ordinal Columns ###

In [10]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

cat_ord_missing_cols = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual',
    'Functional',
]

cat_ord_absent_cols = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

cat_ord_missing_categories = [
    exter_qual_order,
    exter_cond_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
]

cat_ord_absent_categories = [
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Numeric One-Hot-Encoded Columns ###

In [11]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Irrelevant Columns ###

In [12]:
# This col is not informative.

irrelevant_cols = [
    'Id'
]

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# scaling every column after preprocessing decreased accuracy.
# scaling only numeric columns is sufficient.

num_impute_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='median')),
    ('scale',   StandardScaler())
])

num_ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_ord_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=cat_ord_missing_categories)),
])

cat_ord_absent_pipeline = Pipeline([
    ('nan_impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='NA')),
    ('ordinal',     OrdinalEncoder(categories=cat_ord_absent_categories)),
])

cat_ohe_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

cat_ohe_absent_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num_impute',       num_impute_pipeline,       num_impute_cols),
        ('num_ohe',          num_ohe_pipeline,          num_ohe_cols),
        ('cat_ord_absent',   cat_ord_absent_pipeline,   cat_ord_absent_cols),
        ('cat_ord_missing',  cat_ord_missing_pipeline,  cat_ord_missing_cols),
        ('cat_ohe_absent',   cat_ohe_absent_pipeline,   cat_ohe_absent_cols),
        ('cat_ohe_missing',  cat_ohe_missing_pipeline,  cat_ohe_missing_cols),
        ('drop',            'drop',                     irrelevant_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

### Custom Scorer ###

In [14]:
from sklearn.metrics import make_scorer, root_mean_squared_log_error

def safe_rmsle(y_true, y_pred):
    y_pred_pos = np.maximum(y_pred, 0)
    return root_mean_squared_log_error(y_true, y_pred_pos)

safe_rmsle_scorer = make_scorer(safe_rmsle, greater_is_better=False)

### Data Cleanup Pipeline ###

### Full Pipeline (Linear Regression) ###

In [47]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

final_model = TransformedTargetRegressor(
    regressor=final_pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'mean',
        'func': np.log1p,
        'inverse_func': np.expm1
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'mean',
        'func': None,
        'inverse_func': None
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'median',
        'func': np.log1p,
        'inverse_func': np.expm1
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'median',
        'func': None,
        'inverse_func': None
    },
]

### MLFlow Logging (Linear Regression) ###

In [16]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)


mlflow.set_experiment('linear_regression_prep_v1')

for params in param_grid:
    num_imp = params['regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if params['func'] is not None else "rawY"

    run_name = f'LR__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        model = clone(final_model)
        model.set_params(**params)

        cv_results = cross_validate(
            model, X_train, y_train,
            cv=kf,
            scoring={
                'neg_root_mean_squared_log_error': safe_rmsle_scorer,
                'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
                'neg_mean_absolute_error': 'neg_mean_absolute_error',
                'r2': 'r2',
            },
            return_train_score=True
        )

        print(f'Logging parameters ({run_name})')
        for key, value in params.items():
            mlflow.log_param(key, value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))


        print(f'Logging train and validation metrics ({run_name})')

        mlflow.log_metric('mean_train_rmsle',        abs(np.array(cv_results['train_neg_root_mean_squared_log_error']).mean()))
        mlflow.log_metric('mean_validation_rmsle',   abs(np.array(cv_results['test_neg_root_mean_squared_log_error']).mean()))
        mlflow.log_metric('std_train_rmsle',         (np.array(cv_results['train_neg_root_mean_squared_log_error']).std()))
        mlflow.log_metric('std_validation_rmsle',    (np.array(cv_results['test_neg_root_mean_squared_log_error']).std()))

        mlflow.log_metric('mean_train_rmse',        abs(np.array(cv_results['train_neg_root_mean_squared_error']).mean()))
        mlflow.log_metric('mean_validation_rmse',   abs(np.array(cv_results['test_neg_root_mean_squared_error']).mean()))
        mlflow.log_metric('std_train_rmse',         (np.array(cv_results['train_neg_root_mean_squared_error']).std()))
        mlflow.log_metric('std_validation_rmse',    (np.array(cv_results['test_neg_root_mean_squared_error']).std()))

        mlflow.log_metric('mean_train_mae',         abs(np.array(cv_results['train_neg_mean_absolute_error']).mean()))
        mlflow.log_metric('mean_validation_mae',    abs(np.array(cv_results['test_neg_mean_absolute_error']).mean()))
        mlflow.log_metric('std_train_mae',          (np.array(cv_results['train_neg_mean_absolute_error']).std()))
        mlflow.log_metric('std_validation_mae',     (np.array(cv_results['test_neg_mean_absolute_error']).std()))

        mlflow.log_metric('mean_train_r2',          (np.array(cv_results['train_r2']).mean()))
        mlflow.log_metric('mean_validation_r2',     (np.array(cv_results['test_r2']).mean()))
        mlflow.log_metric('std_train_r2',           (np.array(cv_results['train_r2']).std()))
        mlflow.log_metric('std_validation_r2',      (np.array(cv_results['test_r2']).std()))


        for i in range(5):
            mlflow.log_metric(f'split{i}_train_rmsle',       abs(cv_results['train_neg_root_mean_squared_log_error'][i]))
            mlflow.log_metric(f'split{i}_validation_rmsle',  abs(cv_results['test_neg_root_mean_squared_log_error'][i]))
            mlflow.log_metric(f'split{i}_train_rmse',        abs(cv_results['train_neg_root_mean_squared_error'][i]))
            mlflow.log_metric(f'split{i}_validation_rmse',   abs(cv_results['test_neg_root_mean_squared_error'][i]))
            mlflow.log_metric(f'split{i}_train_mae',         abs(cv_results['train_neg_mean_absolute_error'][i]))
            mlflow.log_metric(f'split{i}_validation_mae',    abs(cv_results['test_neg_mean_absolute_error'][i]))
            mlflow.log_metric(f'split{i}_train_r2',          (cv_results['train_r2'][i]))
            mlflow.log_metric(f'split{i}_validation_r2',     (cv_results['test_r2'][i]))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      np.array(cv_results['fit_time']).mean())
        mlflow.log_metric('std_fit_time',       np.array(cv_results['fit_time']).std())
        mlflow.log_metric('mean_score_time',    np.array(cv_results['score_time']).mean())
        mlflow.log_metric('std_score_time',     np.array(cv_results['score_time']).std())

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


Logging parameters (LR__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (LR__prep_v1__logY__num_imp_mean__ord_imp__ohe)
🏃 View run LR__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/5148738ded314a36b3e6831698c18aad
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


KeyboardInterrupt: 

### Full Pipeline (Ridge) ###

In [28]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

final_model = TransformedTargetRegressor(
    regressor=pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
        'regressor__model__alpha': [.001, .01, .1,  1, 5, 9.8, 10, 25, 50, 100, 1000],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
        'regressor__model__alpha': [.001, .01, .1,  1, 5, 9.8, 10, 25, 50, 100, 1000],
        'func': [None],
        'inverse_func': [None],
    },

    # 'regressor__model__alpha': np.linspace(.01, 1000, 300),
    # 'regressor__model__alpha': np.logspace(np.log10(9.5), np.log10(10), 300),
]

### MLFlow Logging (Ridge) ###

In [29]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression_prep_v1')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'RIDGE__alpha_{alpha}__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


Logging parameters (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_mean__ord_imp__ohe)
Logging final model (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_mean__ord_imp__ohe)


2026/04/10 23:25:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_9.8__prep_v1__rawY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/cf0da94b7b724484bd507143d2938fd4
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_median__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_median__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_median__ord_imp__ohe)
Logging final model (RIDGE__alpha_9.8__prep_v1__rawY__num_imp_median__ord_imp__ohe)


2026/04/10 23:28:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_9.8__prep_v1__rawY__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/fdbb9539ff5c483cbb2d36ffb4573aaf
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


### Full Pipeline (Lasso) ###

In [68]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.compose import TransformedTargetRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=100000))
])

final_model = TransformedTargetRegressor(
    regressor=final_pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
        'regressor__model__alpha': [.0001, .0002, .0003, .0004, .0005, .0006, .0007, .0008, .0009, .001, .01, .1, 1, 5, 10],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
        'regressor__model__alpha': [.0001, .0002, .0003, .0004, .0005, .0006, .0007, .0008, .0009, .001, .01, .1, 1, 5, 10],
        'func': [None],
        'inverse_func': [None],
    },

    # 'regressor__model__alpha': np.logspace(-4, 1, 300),
]


### MLFlow Logging (Lasso) ###

In [69]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression_prep_v1')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'LASSO__alpha_{alpha}__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


/home/sandro/.local/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.406e+11, tolerance: 6.360e+08
  model = cd_fast.enet_coordinate_descent(
/home/sandro/.local/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.360e+11, tolerance: 6.361e+08
  model = cd_fast.enet_coordinate_descent(
/home/sandro/.local/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1

Logging parameters (LASSO__alpha_1.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (LASSO__alpha_1.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (LASSO__alpha_1.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (LASSO__alpha_1.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 17:51:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LASSO__alpha_1.0__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/78679e63e535449e866d7a9e688e7b0a
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (LASSO__alpha_1.0__prep_v1__logY__num_imp_median__ord_imp__ohe)
Logging preprocessing parameters (LASSO__alpha_1.0__prep_v1__logY__num_imp_median__ord_imp__ohe)
Logging train and validation metrics (LASSO__alpha_1.0__prep_v1__logY__num_imp_median__ord_imp__ohe)
Logging final model (LASSO__alpha_1.0__prep_v1__logY__num_imp_median__ord_imp__ohe)


2026/04/11 17:53:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run LASSO__alpha_5.0__prep_v1__rawY__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/37fe0bc5bf0a4d78aa3204270e20f184
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (LASSO__alpha_10.0__prep_v1__rawY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (LASSO__alpha_10.0__prep_v1__rawY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (LASSO__alpha_10.0__prep_v1__rawY__num_imp_mean__ord_imp__ohe)
Logging final model (LASSO__alpha_10.0__prep_v1__rawY__num_imp_mean__ord_imp__ohe)


2026/04/11 19:17:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LASSO__alpha_10.0__prep_v1__rawY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/09629d3d5fa8404c9beedcebf3a096b2
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (LASSO__alpha_10.0__prep_v1__rawY__num_imp_median__ord_imp__ohe)
Logging preprocessing parameters (LASSO__alpha_10.0__prep_v1__rawY__num_imp_median__ord_imp__ohe)
Logging train and validation metrics (LASSO__alpha_10.0__prep_v1__rawY__num_imp_median__ord_imp__ohe)
Logging final model (LASSO__alpha_10.0__prep_v1__rawY__num_imp_median__ord_imp__ohe)


2026/04/11 19:19:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LASSO__alpha_10.0__prep_v1__rawY__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/c1c531bd8086465489029d5f2d49ae45
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


### Final Pipeline (ElasticNet) ###

In [75]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import KFold
from sklearn.compose import TransformedTargetRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', ElasticNet(max_iter=10000)),
])

final_model = TransformedTargetRegressor(
    regressor=final_pipeline,
)

param_grid = {
    'regressor__preprocessor__num_impute__impute__strategy': ['mean'],
    'regressor__model__alpha': [.0005, .0007, .001, .005, .1, 1],
    'regressor__model__l1_ratio': [.1, .3, .5, .6, .7, .8, .9],
    'func': [np.log1p],
    'inverse_func': [np.expm1],
}

### MLFlow Logging (Elastic Net) ###

In [76]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression_prep_v1')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    l1_ratio = row_dict['param_regressor__model__l1_ratio']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'ELASTIC_NET__alpha_{alpha}__l1_ratio_{l1_ratio}__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


Logging parameters (ELASTIC_NET__alpha_0.0005__l1_ratio_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (ELASTIC_NET__alpha_0.0005__l1_ratio_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (ELASTIC_NET__alpha_0.0005__l1_ratio_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (ELASTIC_NET__alpha_0.0005__l1_ratio_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 21:41:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run ELASTIC_NET__alpha_0.0005__l1_ratio_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/1ca81bb494e54fe78ccaac29c66319d3
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (ELASTIC_NET__alpha_0.0005__l1_ratio_0.3__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (ELASTIC_NET__alpha_0.0005__l1_ratio_0.3__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (ELASTIC_NET__alpha_0.0005__l1_ratio_0.3__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (ELASTIC_NET__alpha_0.0005__l1_ratio_0.3__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 21:43:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run ELASTIC_NET__alpha_0.0005__l1_ratio_0.3__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/5f5ca82d119b4775bede380d84ee6165
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (ELASTIC_NET__alpha_0.0005__l1_ratio_0.5__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (ELASTIC_NET__alpha_0.0005__l1_ratio_0.5__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (ELASTIC_NET__alpha_0.0005__l1_ratio_0.5__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (ELASTIC_NET__alpha_0.0005__l1_ratio_0.5__prep_v1__logY__num_imp_mean__ord_imp__ohe)


...

🏃 View run ELASTIC_NET__alpha_1.0__l1_ratio_0.7__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/510b76c2a9d7478ba70c98db51f66557
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (ELASTIC_NET__alpha_1.0__l1_ratio_0.8__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (ELASTIC_NET__alpha_1.0__l1_ratio_0.8__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (ELASTIC_NET__alpha_1.0__l1_ratio_0.8__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (ELASTIC_NET__alpha_1.0__l1_ratio_0.8__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 23:20:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run ELASTIC_NET__alpha_1.0__l1_ratio_0.8__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/7d983b88e945432a81687da209e94c44
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
Logging parameters (ELASTIC_NET__alpha_1.0__l1_ratio_0.9__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (ELASTIC_NET__alpha_1.0__l1_ratio_0.9__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (ELASTIC_NET__alpha_1.0__l1_ratio_0.9__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (ELASTIC_NET__alpha_1.0__l1_ratio_0.9__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 23:23:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run ELASTIC_NET__alpha_1.0__l1_ratio_0.9__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/f6420c9939304799a2209e43d4b4cd2e
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


### Decision Tree ###

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=1337))
])

final_model = TransformedTargetRegressor(
    regressor=final_pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean'],
        'regressor__model__max_depth': [5, 6, 7, 8, 9, 10, 15, 20, 25, None],
        'regressor__model__min_samples_leaf': [1],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean'],
        'regressor__model__max_depth': [5, 10, 15],
        'regressor__model__min_samples_leaf': [5, 10, 15, 20],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },
]



### MLFlow Logging (Decision Tree) ###

In [19]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('decision_tree_prep_v1')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    max_depth = row_dict['param_regressor__model__max_depth']
    min_sample_leaf = row_dict['param_regressor__model__min_samples_leaf']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'DT__max_depth_{max_depth}__min_samples_leaf_{min_sample_leaf}__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'DecisionTree')
        mlflow.set_tag('model_id', model_info.model_id)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Logging parameters (decision_tree__max_depth_1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (decision_tree__max_depth_1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (decision_tree__max_depth_1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (decision_tree__max_depth_1__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/14 10:41:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.2623952250679163 47168.664474467565 35003.69176788938 0.4885785238282333
🏃 View run DT__max_depth_1__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/6/runs/d85347927b374776a0703dbb13b0befb
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/6


### Final Test scores (Check Locally That It Works) ###

In [16]:
model_id = 'm-bfb97a211a5246268ddefd7eba744eb2'
model = mlflow.sklearn.load_model(f'models:/{model_id}')

y_pred = model.predict(X_test)
print(y_pred[:4])
print(f'test_rmsle: {root_mean_squared_log_error(y_test, y_pred)}')

[221852.82560538 312064.25197263 115527.50108275  69858.81023941]
test_rmsle: 0.12809132069393636


### Final Test scores (Upload On MLFlow) ###

In [35]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

runs_df = mlflow.search_runs(experiment_names=["bagging_prep_v3",])

for _, row in runs_df.iterrows():
    run_id = row["run_id"]
    model_id = row["tags.model_id"]
    # model_id = 'm-a7c40163c9094157be2aa2bd43a85ef4'
    print('model_id:', model_id)

    model = mlflow.sklearn.load_model(f"models:/{model_id}")
    y_pred = model.predict(X_test)

    test_rmsle  = safe_rmsle(y_test, y_pred)
    test_rmse   = root_mean_squared_error(y_test, y_pred)
    test_mae    = mean_absolute_error(y_test, y_pred)
    test_r2     = r2_score(y_test, y_pred)
    print(test_rmsle, test_rmse, test_mae, test_r2)
    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric('test_rmsle', test_rmsle)
        mlflow.log_metric('test_rmse',  test_rmse)
        mlflow.log_metric('test_mae',   test_mae)
        mlflow.log_metric('test_r2',    test_r2)

model_id: m-5fc08a5d7b4e49ff88f521e5b8234def


0.1127532386298078 18625.72238908595 13435.437091005473 0.9202559932707391
🏃 View run BAGGING__LASSO__alpha_0.0002__n_estimators__250__max_samples_0.7__max_features_0.4__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/934d7c0ac82647f599e75b22deca36c1
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-7a98bbacc2a14a61be03e7e00df75843


0.11332703818058204 18824.839253215236 13564.09427824721 0.9185418852746993
🏃 View run BAGGING__LASSO__alpha_0.0002__n_estimators__150__max_samples_0.7__max_features_0.4__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/80337e4deb5b4847ab00ac5b257662a8
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-63c6911f649b447399857951b1d0aebd


0.11272471706017097 18632.639642711 13435.77874195384 0.9201967513324179
🏃 View run BAGGING__LASSO__alpha_0.0002__n_estimators__250__max_samples_0.6__max_features_0.4__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/e775508dda5d475db8365fa1d2ee0086
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-810a6ee58f7348808aeb3d0bbe9bfb88


0.11328570967692406 18787.854866323858 13537.359391187876 0.9188616456665198
🏃 View run BAGGING__LASSO__alpha_0.0002__n_estimators__150__max_samples_0.6__max_features_0.4__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/c21057c0e15d4c919b144a714b4b268c
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-c6b1cf3dec1a41cfb536880a564b9ce1


...

0.11322068573849614 18746.60193345945 13497.980393852502 0.9192175692512532
🏃 View run BAGGING__LASSO__alpha_0.0001__n_estimators__150__max_samples_0.6__max_features_0.4__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/541bbce463fc447890c72347d9653f52
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-94b3dbb04fea46f1b340411664430d0f


0.1147233787311203 18977.911032769996 13675.568276351069 0.9172117667856443
🏃 View run BAGGING__LASSO__alpha_0.0001__n_estimators__250__max_samples_0.7__max_features_0.3__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/171b50456fa149ca996d531ecbd7b960
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-80b37512b4ea4d329571243164ee1c6d


0.1150408788698737 19019.05341456487 13742.632447120435 0.91685242300372
🏃 View run BAGGING__LASSO__alpha_0.0001__n_estimators__150__max_samples_0.7__max_features_0.3__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/5a3b5e3e8d214ede835c9fb295007fc9
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-da45e9064b0c428a867262c465a791ac


0.11466303581885957 18947.97669106926 13677.613955423401 0.917472728783489
🏃 View run BAGGING__LASSO__alpha_0.0001__n_estimators__250__max_samples_0.6__max_features_0.3__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/8adeab8eee174f6f988a128c8567ad50
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
model_id: m-5fcfad1e87304fd09ae4bfaae3269177


0.11500038715375623 18947.313339359647 13726.126372453245 0.9174785070941829
🏃 View run BAGGING__LASSO__alpha_0.0001__n_estimators__150__max_samples_0.6__max_features_0.3__prep_v3__logY__num_imp_mean__ord_imp__ohe__no_outliers__corr_threshold_1__k_features_all__feature_selector_score_fn_f_regression at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8/runs/06e2545b4c7e47a1b9dc743aa5049d75
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/8
